# Model Evaluation & Feature Importance
This notebook evaluates the performance of the multi-task LSTM model on the unseen test set and analyzes feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, \
    classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

# Settings
plt.style.use('ggplot')
os.makedirs('../models/evaluation', exist_ok=True)

## Section 1 — Load Everything

In [ ]:
model = tf.keras.models.load_model('../models/trained/best_model.keras')

prep_dir = Path('../models/preprocessing')
X_test = np.load(prep_dir / 'X_test.npy')
y_demand_test = np.load(prep_dir / 'y_demand_test.npy')
y_anomaly_test = np.load(prep_dir / 'y_anomaly_test.npy')
y_alert_test = np.load(prep_dir / 'y_alert_test.npy')

scaler = joblib.load(prep_dir / 'scaler.pkl')

print('Predicting on test set...')
preds = model.predict(X_test, verbose=1)
p_demand, p_anomaly, p_alert = preds

print(f'p_demand shape: {p_demand.shape}')
print(f'p_anomaly shape: {p_anomaly.shape}')
print(f'p_alert shape: {p_alert.shape}')

## Section 2 — Demand Prediction Evaluation

In [ ]:
# We need to inverse transform demand. 
# Note: y_demand_test is raw kW, but the model predicted scaled values? 
# Wait, my training script used raw kW as targets. 
# So p_demand is already in kW (approximately). 
# Actually, I should check if I scaled the targets during preprocessing. 
# In create_sequences: y_demand.append(raw_df['total_hospital_kw'].iloc[...].values)
# So y_demand is raw kW. p_demand is also raw kW.

y_true = y_demand_test
y_pred = p_demand

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

print(f'MAE: {mae:.2f} kW')
print(f'RMSE: {rmse:.2f} kW')
print(f'R2 Score: {r2:.4f}')
print(f'MAPE: {mape:.2f}%')

plt.figure(figsize=(15, 6))
plt.plot(y_true[:48*7, 0], label='Actual (Next 30min)', alpha=0.8)
plt.plot(y_pred[:48*7, 0], label='Predicted (Next 30min)', alpha=0.8)
plt.title('Demand Prediction (First week of Test Set)')
plt.legend()
plt.savefig('figures/eval_demand_timeline.png')
plt.show()

## Section 3 — Anomaly Detection Evaluation

In [ ]:
threshold = 0.3
y_true_anom = y_anomaly_test
y_pred_anom = (p_anomaly > threshold).astype(int)

print(classification_report(y_true_anom, y_pred_anom, target_names=['Normal', 'Outage']))

cm = confusion_matrix(y_true_anom, y_pred_anom)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Anomaly Detection Confusion Matrix')
plt.savefig('figures/eval_anomaly_cm.png')
plt.show()

fpr, tpr, _ = roc_curve(y_true_anom, p_anomaly)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.savefig('figures/eval_anomaly_roc.png')
plt.show()

## Section 4 — Alert Classification Evaluation

In [ ]:
y_true_alert = y_alert_test
y_pred_alert = np.argmax(p_alert, axis=1)
alert_names = ['NORMAL', 'WARNING', 'CRITICAL']

print(classification_report(y_true_alert, y_pred_alert, target_names=alert_names))

cm_alert = confusion_matrix(y_true_alert, y_pred_alert)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_alert, annot=True, fmt='d', cmap='YlOrRd', xticklabels=alert_names, yticklabels=alert_names)
plt.title('Alert Classification Confusion Matrix')
plt.savefig('figures/eval_alert_cm.png')
plt.show()

## Section 6 — Feature Importance (Permutation)

In [ ]:
print('Computing permutation importance (Demand MAE)...')
baseline_preds = model.predict(X_test, verbose=0)[0]
baseline_mae = mean_absolute_error(y_demand_test, baseline_preds)

importances = []
features = [
    'net_solar_kw', 'net_wind_kw', 'grid_available_kw', 'total_supply_kw',
    'total_hospital_kw', 'energy_balance_kw', 'avg_battery_pct', 'min_battery_pct',
    'temperature_2m', 'cloud_cover', 'windspeed_10m', 'precipitation',
    'hour', 'is_weekend', 'is_daytime', 'renewable_fraction', 'is_trading',
    'is_winter', 'is_spring', 'is_summer', 'is_autumn'
]

for i in range(X_test.shape[2]):
    X_perm = X_test.copy()
    np.random.shuffle(X_perm[:, :, i])
    perm_preds = model.predict(X_perm, verbose=0)[0]
    perm_mae = mean_absolute_error(y_demand_test, perm_preds)
    importances.append(perm_mae - baseline_mae)
    print(f'Finished {features[i]}')

imp_df = pd.DataFrame({'feature': features, 'importance': importances}).sort_values('importance', ascending=True)
plt.figure(figsize=(10, 12))
plt.barh(imp_df['feature'], imp_df['importance'])
plt.title('Feature Importance (Increase in MAE when shuffled)')
plt.savefig('figures/eval_importance.png')
plt.show()

## Section 7 — Final Scorecard

In [ ]:
y_true_anom = y_anomaly_test
y_pred_anom = (p_anomaly > 0.3).astype(int)
recall_anom = np.sum((y_true_anom == 1) & (y_pred_anom == 1)) / (np.sum(y_true_anom == 1) + 1e-9)

y_true_alert = y_alert_test
y_pred_alert = np.argmax(p_alert, axis=1)
critical_recall = np.sum((y_true_alert == 2) & (y_pred_alert == 2)) / (np.sum(y_true_alert == 2) + 1e-9)

scorecard = {
    "demand": {"MAE_kw": float(mae), "RMSE_kw": float(rmse), "R2": float(r2), "MAPE_pct": float(mape)},
    "anomaly": {"recall": float(recall_anom), "auc": float(roc_auc)},
    "alert": {"critical_recall": float(critical_recall)},
    "overall_grade": "PASS" if (r2 > 0.8 and recall_anom > 0.85 and critical_recall > 0.80) else "FAIL"
}

print('Final Scorecard:')
print(json.dumps(scorecard, indent=2))

with open('../models/evaluation/scorecard.json', 'w') as f:
    json.dump(scorecard, f, indent=2)